In [1]:
import torch

In [4]:
W = torch.rand((384, 3, 14, 14))

In [7]:
sigma = torch.tensor([58.395, 57.12, 57.375])  # then reshape it

In [11]:
sigma_reshaped = sigma.reshape(3,1,1)

In [12]:
W_hat = W / sigma_reshaped

In [13]:
W_hat

tensor([[[[6.6887e-03, 1.5344e-02, 1.4230e-03,  ..., 1.6552e-02,
           7.8995e-03, 3.5881e-03],
          [1.6460e-02, 1.1785e-02, 5.1060e-03,  ..., 1.6321e-02,
           4.8128e-03, 9.0574e-03],
          [8.3085e-03, 1.4382e-02, 1.6435e-02,  ..., 3.1911e-03,
           7.1477e-03, 1.4298e-02],
          ...,
          [1.4459e-02, 1.2434e-02, 1.6823e-02,  ..., 1.2946e-02,
           1.4641e-02, 9.1205e-03],
          [1.1047e-02, 4.9683e-03, 1.3411e-02,  ..., 3.9311e-03,
           9.2987e-03, 1.1637e-02],
          [1.0683e-02, 1.3272e-02, 1.5543e-02,  ..., 9.4441e-03,
           7.2950e-03, 1.5932e-02]],

         [[9.7597e-03, 6.0725e-03, 1.6362e-04,  ..., 1.1590e-02,
           1.3012e-02, 5.5547e-03],
          [8.9806e-03, 5.0193e-03, 1.4145e-02,  ..., 8.5639e-03,
           1.6608e-02, 5.4106e-03],
          [2.1704e-03, 1.7009e-02, 7.7462e-03,  ..., 1.1111e-02,
           1.5447e-02, 5.6761e-03],
          ...,
          [6.2381e-03, 2.0651e-03, 1.4680e-02,  ..., 4.0157

xFormers not available
xFormers not available


TypeError: DepthAnythingV2.__init__() got an unexpected keyword argument 'max_depth'

In [3]:
import torch
from depth_anything_v2.dpt import DepthAnythingV2

torch.manual_seed(0)

# ---- load the real model (relative, ViT-S) ----
model = DepthAnythingV2(encoder='vits', features=64,
                        out_channels=[48, 96, 192, 384])
model.load_state_dict(torch.load(
    'checkpoints/depth_anything_v2_vits.pth', map_location='cpu'))
model.eval()

conv = model.pretrained.patch_embed.proj
print(conv)   # expect: Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))

# ---- normalization constants (same as in dpt.py forward) ----
mu    = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
sigma = torch.tensor([58.395,  57.12,  57.375]).view(3, 1, 1)

with torch.no_grad():
    W = conv.weight          # (384, 3, 14, 14)
    b = conv.bias            # (384,)

    # ---- fusion ----
    W_prime = W / sigma                                   # divide along input-channel dim
    b_prime = b - (W * mu / sigma).sum(dim=(1, 2, 3))     # (384,)

    fused_conv = torch.nn.Conv2d(3, 384, kernel_size=14, stride=14)
    fused_conv.weight.copy_(W_prime)
    fused_conv.bias.copy_(b_prime)

    # ---- verification ----
    x_raw = torch.rand(1, 3, 322, 434) * 255              # raw image range [0, 255]

    ref = conv((x_raw - mu) / sigma)                      # original: normalize, then conv
    out = fused_conv(x_raw)                               # fused: conv only

    abs_err = (ref - out).abs().max().item()
    rel_err = abs_err / ref.abs().max().item()
    print(f"output shape : {tuple(out.shape)}")           # (1, 384, 23, 31)
    print(f"max abs error: {abs_err:.2e}")
    print(f"max rel error: {rel_err:.2e}")
    print("PASS" if rel_err < 1e-5 else "FAIL")

Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
output shape : (1, 384, 23, 31)
max abs error: 2.03e-06
max rel error: 1.51e-06
PASS
